In [ ]:
%%bash
SRC=$(find /kaggle/input -name kaggle_setup.sh -print -quit)
[ -n "$SRC" ] || { echo "ERROR: không thấy kaggle_setup.sh, dataset code đã gắn chưa?"; exit 1; }
QW=$(grep -rlE --include=config.json '"model_type"[[:space:]]*:[[:space:]]*"qwen3"' /kaggle/input 2>/dev/null | head -1 | xargs -r dirname)
PH=$(grep -rlE --include=config.json '"model_type"[[:space:]]*:[[:space:]]*"whisper"' /kaggle/input 2>/dev/null | head -1 | xargs -r dirname)
echo "Qwen3 dir  : ${QW:-KHÔNG THẤY}"; echo "Whisper dir: ${PH:-KHÔNG THẤY}"
[ -n "$QW" ] && [ -n "$PH" ] || { echo "ERROR: thiếu thư mục model"; exit 1; }
cp "$SRC" /kaggle/working/setup_patched.sh
sed -i -e "s#^QWEN=.*#QWEN=$QW#" -e "s#^PHO=.*#PHO=$PH#" -e 's/python -I /python /g' /kaggle/working/setup_patched.sh
bash /kaggle/working/setup_patched.sh
sed -i 's/python -I /python /g' /kaggle/working/lifemate/scripts/*.sh
echo "--- đã khôi phục?"; ls /kaggle/working/checkpoints /kaggle/working/checkpoints/q4b_lora | head

In [ ]:
%%bash
pip uninstall -y torchao 2>&1 | tail -2
python -c "import peft, transformers; print('peft', peft.__version__, '| transformers', transformers.__version__, '| import OK')"

In [ ]:
%%bash
# B3: kiểm tra đã có đủ thứ để chạy public_test chưa
source /kaggle/working/lifemate/kaggle_env.sh
echo "== adapter LLM"; ls checkpoints/q4b_lora | head
echo "== ASR public_test"; ls -la data/processed | grep -E "asr_.*public_test" || echo "KHÔNG CÓ asr_*_public_test -> chạy cell B4"
echo "== số mẫu public_test"; wc -l "data/raw/train+public_test/public_test/public_test.jsonl"

In [ ]:
%%bash
# B4 (CHỈ CHẠY KHI B3 báo thiếu asr_phowl_ft_public_test.jsonl): ASR đã fine-tune cho public_test
source /kaggle/working/lifemate/kaggle_env.sh
ls checkpoints/phowl_ft/merged | head -3
FORCE=1 bash scripts/kaggle_pipeline.sh asr_ft
ls -la data/processed | grep -E "asr_.*public_test"

In [ ]:
%%bash
# B5: suy luận trên public_test (greedy, tất định) -> outputs/submission_public.jsonl
source /kaggle/working/lifemate/kaggle_env.sh
bash scripts/infer_multi_gpu.sh public_test outputs/submission_public.jsonl --backend hf --model $QWEN --adapter checkpoints/q4b_lora --text asr:phowl_ft
wc -l outputs/submission_public.jsonl
echo "== thời gian mỗi mẫu (dòng cuối mỗi shard)"; tail -qn 1 outputs/submission_public.part*.log

In [ ]:
%%bash
# B6: kiểm tra schema bằng công cụ của BTC (phải chạy trong thư mục public_test)
source /kaggle/working/lifemate/kaggle_env.sh
cd "data/raw/train+public_test/public_test"
python -X utf8 -m zaic_tools --split public_test --root .. --validate /kaggle/working/lifemate/outputs/submission_public.jsonl | tail -15

In [ ]:
%%bash
# B7: đóng gói file nộp + thống kê nhanh
source /kaggle/working/lifemate/kaggle_env.sh
cp outputs/submission_public.jsonl /kaggle/working/submission.jsonl
python -X utf8 - <<'EOF'
import json
rows = [json.loads(l) for l in open('/kaggle/working/submission.jsonl', encoding='utf-8')]
ids = [r['id'] for r in rows]
ref = [json.loads(l)['id'] for l in open('data/raw/train+public_test/public_test/public_test.jsonl', encoding='utf-8')]
print('dòng:', len(rows), '| id duy nhất:', len(set(ids)), '| đủ id public_test:', set(ids) == set(ref))
print('rỗng (oos):', sum(1 for r in rows if not r['tool_calls']), '| ask_missing_info:', sum(1 for r in rows if any(c['name'] == 'sys.ask_missing_info' for c in r['tool_calls'])))
EOF
sha256sum /kaggle/working/submission.jsonl
ls -la /kaggle/working/submission.jsonl